# Day 03 — Chunking and document structure: where you cut decides what retrieval can find

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-03/LEARN.md`](../docs/day-03/LEARN.md).*

A retriever never returns a document. It returns a **chunk**: a piece of one. If the cut separates a table row from its column headers, or an answer from the sentence that explains it, the retriever can find the right place and still hand over something that doesn't answer the question.

Today we cut Day 02's 11 records in two ways and test both on the same 13 questions:

```
 11 Document records ──┬──> A. fixed-size windows: every 40 words, wherever that falls
 (## headings,         │
  table rows)          └──> B. structure-aware: at headings, never inside a sentence or a table row,
                                with "Title > Section" on top
            both ──> Day 01's BM25 retriever, top 3 ──> is the whole answer inside one retrieved chunk?
```

| This notebook **does** | This notebook **does not** |
|---|---|
| Split documents by word count and by structure, and show exactly where each one cuts | Use embeddings or a vector database (Days 04–05) |
| Keep tables whole, or split them between rows with the header repeated | Generate answers with an LLM (Day 06) |
| Compare both methods on 13 questions, including the Singapore hotel-limit table question | Claim a benchmark: the questions are diagnostic, not a sample (Day 07 builds the eval set) |
| Measure what overlap, window size and section headers really change | Enforce permissions (Days 05 and 14), although every chunk carries them |
| Show why stable chunk IDs make updates and citations cheaper | Call any paid API or need internet access |

**Cost:** free. **Runtime:** a few seconds. **Requirements:** Python 3.11+, Jupyter and `pypdf` (all pinned in `requirements.txt`). Nothing new to install today.
**How to run:** open this notebook from inside the cloned repository, then *Kernel → Restart Kernel and Run All Cells*.
**Verified:** executed top to bottom on 2026-10-15 on Linux with Python 3.11, 3.12, 3.13 and 3.14. The saved outputs come from the Python 3.13 run. CI runs the same notebook on Linux, Windows and macOS on every push.

## 0. Setup — reuse Day 01 and Day 02 without copying them

Day 03 needs two things built earlier: Day 02's **ingestion pipeline** (to get the records) and Day 01's **BM25 retriever** (to test the chunks). Pasting both here would add about 350 lines that have nothing to do with chunking. Instead they live in a small shared package, [`src/rag_from_scratch/`](../src/rag_from_scratch/), and this notebook imports it.

- The Day 01 and Day 02 notebooks still contain the full, explained code. The package is a copy for reuse, not a replacement.
- `tests/test_shared_code.py` runs those two notebooks' own code and checks that the package returns *identical* records and scores, so the copies cannot drift apart.
- There is nothing to install: the next cell adds `src/` to Python's import path for this session only.

In [1]:
import json
import platform
import re
import statistics
import sys
from dataclasses import asdict, dataclass, replace
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))          # import the shared code without installing it

from rag_from_scratch.bm25 import BM25Index            # Day 01's retriever
from rag_from_scratch.ingest import ingest, sha256     # Day 02's pipeline

print(f"Python {platform.python_version()}")

Python 3.13.16


In [2]:
WINDOW_WORDS = 40    # fixed-size chunks: words per window (close to the median section length, section 1)
OVERLAP_WORDS = 10   # section 7: words that neighbouring windows share (25 %)
MAX_WORDS = 80       # structure-aware chunks: a longer section is split (section 4.1)
TOP_K = 3            # chunks retrieved per question, as on Day 01

## 1. The input — Day 02's records

Day 02 turned 13 files into 11 `Document` records. Whatever the source format, every body uses the same convention: `## ` starts a section, and a table row is one line with cells joined by ` | `. That convention is what makes structure-aware chunking possible today.

We measure size in **words**: a word here starts with a letter or digit, so table pipes and heading marks don't count. Real systems count **tokens** with their embedding model's tokenizer, because models have token limits. Words keep this notebook dependency-free; LEARN.md explains how to switch.

In [3]:
documents, quarantined = ingest(REPO_ROOT)
by_doc = {d.doc_id: d for d in documents}

WORD = re.compile(r"\w[^\s|]*")      # starts with a letter or digit; "|", "##" and ">" are not words


def count_words(text: str) -> int:
    return len(WORD.findall(text))


def is_table_row(line: str) -> bool:
    return " | " in line             # Day 02's convention for table rows


def split_sections(body: str) -> list[tuple[str, list[str]]]:
    # Day 01's rule: "## " starts a section; text before the first heading becomes "Overview",
    # so nothing is silently dropped.
    sections, heading, lines = [], "Overview", []
    for line in body.splitlines():
        if line.startswith("## "):
            if lines:
                sections.append((heading, lines))
            heading, lines = line[3:].strip(), []
        elif line.strip():
            lines.append(line)
    if lines:
        sections.append((heading, lines))
    return sections


print(f"{len(documents)} documents from Day 02's pipeline ({len(quarantined)} files stay quarantined)\n")
print(f"{'doc_id':<29}{'words':>6}  {'words per section':<24}table rows")
section_sizes = []
for d in documents:
    sizes = [count_words("\n".join(lines)) for _, lines in split_sections(d.body)]
    section_sizes += sizes
    rows = sum(is_table_row(line) for line in d.body.splitlines())
    print(f"{d.doc_id:<29}{count_words(d.body):>6}  {' '.join(map(str, sizes)):<24}{rows or '-'}")

print(f"\n{len(section_sizes)} sections, {min(section_sizes)}-{max(section_sizes)} words each "
      f"(median {statistics.median(section_sizes):.0f})")

11 documents from Day 02's pipeline (2 files stay quarantined)

doc_id                        words  words per section       table rows
data-retention-policy           109  27 31 24 20             -
expense-reimbursement-policy    133  20 36 23 29 16          -
it-software-requests            132  31 37 37 17             5
manager-pay-review-guide         68  68                      -
new-starter-it-checklist         88  29 23 13 14             -
parental-leave-policy           114  24 34 24 20             -
password-and-mfa-standard       123  27 41 20 27             -
remote-work-policy              179  30 39 25 45 29          -
security-incident-reporting     137  36 26 30 30             -
travel-policy                   208  28 41 58 32 36          5
vpn-access-guide                127  33 26 28 30             -

44 sections, 13-68 words each (median 29)


Small documents, short sections, and two tables: one rebuilt from a PDF (`travel-policy`), one from an HTML page (`it-software-requests`). Real handbooks are longer, but the problems below are the same at any scale; what matters is the size of a chunk *relative to* the structure it cuts through.

## 2. What makes a good chunk?

Before comparing methods, agree on the yardstick. A chunk is the unit that gets **retrieved, shown to a reader or a model, cited, filtered by permissions and updated**. So a good chunk:

1. **Answers on its own.** Read in isolation, it contains the whole fact *and* what the fact means (a number with its column header, a rule with its subject).
2. **Says where it came from.** Document and section, so a reader can judge it and a citation can point to it.
3. **Is the right size.** Small enough to be specific, big enough to keep its context. Every retrieved word costs prompt space.
4. **Keeps its document's metadata.** Permissions, source and version travel with every chunk.
5. **Has a stable ID.** An edit in one section shouldn't rename or change every other chunk.

Each section below checks these properties with code, not by eye.

## 3. Method A — fixed-size windows

The simplest chunker counts: every `WINDOW_WORDS` words, cut. It needs no knowledge of the document, works on any text, and produces chunks of predictable size. That is why it is the default in many tutorials.

Each chunk records the metadata it inherits from its document (point 4 above) and a hash of its own text, which section 8 uses.

In [4]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: str                  # stable, citable ID
    doc_id: str
    section: str | None            # heading the text came from (None for fixed-size windows)
    text: str                      # exactly what is indexed, retrieved and shown to a reader
    # Inherited from the Document, so every retrieved chunk can be filtered, cited and dated
    title: str
    allowed_groups: tuple[str, ...]
    source_path: str
    last_reviewed: str | None
    doc_hash: str                  # the document version this chunk was cut from
    chunk_hash: str                # this chunk's own version (section 8)


def make_chunk(doc, chunk_id: str, section: str | None, text: str) -> Chunk:
    return Chunk(chunk_id, doc.doc_id, section, text, doc.title, doc.allowed_groups, doc.source_path,
                 doc.last_reviewed, doc.content_hash, sha256(text))


def fixed_size_chunks(doc, size: int | None = None, overlap: int = 0) -> list[Chunk]:
    # Windows of `size` words; each starts `size - overlap` words after the previous one.
    # The cut falls wherever the count says: mid-sentence, mid-table, after a heading.
    size = WINDOW_WORDS if size is None else size
    assert 0 <= overlap < size
    spans = [m.span() for m in WORD.finditer(doc.body)]        # (start, end) of every word
    chunks = []
    for first in range(0, len(spans), size - overlap):
        window = spans[first:first + size]
        text = doc.body[window[0][0]:window[-1][1]]           # a slice of the body: line breaks survive
        chunks.append(make_chunk(doc, f"{doc.doc_id}#w{first:03d}", None, text))
        if first + size >= len(spans):
            break
    return chunks


fixed = [c for d in documents for c in fixed_size_chunks(d)]
print(f"{len(documents)} documents -> {len(fixed)} fixed-size chunks of up to {WINDOW_WORDS} words")
print("travel-policy:", ", ".join(c.chunk_id.split("#")[1] for c in fixed if c.doc_id == "travel-policy"))

11 documents -> 42 fixed-size chunks of up to 40 words
travel-policy: w000, w040, w080, w120, w160, w200


The IDs say *where* a chunk starts (`w080` = the window starting at word 80), not *what* it is about. Section 8 shows why that matters when a document changes.

Now look at the table in the travel policy. The function below prints every line of a document next to the chunk(s) that hold it. A line held by **two** chunks was cut in half (✂).

In [5]:
def boundary_map(doc, chunks: list[Chunk], lines_wanted: slice = slice(None)) -> None:
    # For each line of the body: which windows contain its words? Two or more = the line was cut.
    spans = [m.span() for m in WORD.finditer(doc.body)]
    windows = []
    for c in chunks:
        first = int(c.chunk_id.rsplit("#w", 1)[1])
        start, end = spans[first][0], spans[first][0] + len(c.text)
        windows.append((c.chunk_id.split("#")[1], start, end))
    offset, rows = 0, []
    for line in doc.body.split("\n"):
        start, end = offset, offset + len(line)
        offset = end + 1
        if line.strip():
            holders = [name for name, s, e in windows if s < end and e > start]
            rows.append((holders, line))
    for holders, line in rows[lines_wanted]:
        label = " ✂ ".join(holders) if len(holders) > 1 else holders[0] if holders else "-"
        print(f"{label:<11}  {line if len(line) <= 82 else line[:79] + '...'}")


travel = by_doc["travel-policy"]
boundary_map(travel, fixed_size_chunks(travel))

w000         ## Purpose
w000         This policy explains how Acme employees book business travel and claim reimburs...
w000         ## Booking travel
w000 ✂ w040  Book flights, trains and hotels through the travel portal at least 14 days befo...
w040         ## Daily limits by city
w040 ✂ w080  Hotel and meal costs are reimbursed up to the daily limits below. Limits are pe...
w080         City | Hotel per night | Meals per day
w080         London | USD 250 | USD 75
w080 ✂ w120  New York | USD 300 | USD 75
w120         Singapore | USD 220 | USD 60
w120         Other cities | USD 180 | USD 50
w120         ## Claiming travel costs
w120 ✂ w160  Claim travel costs in the expense system, as described in the Expense Reimburse...
w160         ## Safety while travelling
w160 ✂ w200  Register every international trip in the travel portal, so the Security Team ca...


Read the middle of that map. The boundary between `w080` and `w120` falls **inside the table**, in the middle of the New York row. The header row (`City | Hotel per night | Meals per day`) and London stay in `w080`; Singapore and Other cities land in `w120`. Here is the chunk that holds the Singapore row:

In [6]:
by_fixed = {c.chunk_id: c for c in fixed}


def show(chunk: Chunk) -> None:
    print(f"[{chunk.chunk_id}]  {count_words(chunk.text)} words")
    for line in chunk.text.splitlines():
        print("   ", line)
    print()


singapore_fixed = next(c for c in fixed if "Singapore | USD 220" in c.text)
show(singapore_fixed)
print("Contains the column header 'Hotel per night'?", "Hotel per night" in singapore_fixed.text)

[travel-policy#w120]  40 words
    300 | USD 75
    Singapore | USD 220 | USD 60
    Other cities | USD 180 | USD 50
    
    ## Claiming travel costs
    Claim travel costs in the expense system, as described in the Expense Reimbursement Policy, and attach itemised receipts. Costs above the daily limits

Contains the column header 'Hotel per night'? False


`Singapore | USD 220 | USD 60`. Is the hotel limit 220 or 60? The chunk can't say: the row that names the columns is in another chunk. It even starts with `300 | USD 75`, the tail of the New York row, which on its own means nothing. A reader, or a language model on Day 06, would have to guess.

## 4. Method B — structure-aware chunks

The second chunker uses the structure that Day 02 worked hard to keep:

1. **Split at headings.** One section is the starting unit, as on Day 01.
2. **Never cut inside a unit.** A paragraph or list item is a unit; a whole table (consecutive ` | ` rows) is a unit.
3. **Pack units up to `MAX_WORDS`.** A section that is too long is split *between* units. A table that is too long on its own is split between rows, and **each piece repeats the header row**. A paragraph that is too long is split between sentences.
4. **Add a context header**: `Title > Section` on the first line, so the chunk says where it came from.

IDs come from the heading (`travel-policy#daily-limits-by-city`), exactly as on Day 01; a section that has to be split gets `-part-1`, `-part-2`, and so on. Assumption to keep in mind: the first row of a table is its header, which holds for both tables here but not for every table you will meet.

In [7]:
SENTENCE_END = re.compile(r"(?<=[.!?])\s+")


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


def to_units(lines: list[str]) -> list[list[str]]:
    # Units that must not be cut: a run of table rows is one unit; any other line
    # (a paragraph or a list item) is a unit of its own.
    units: list[list[str]] = []
    for line in lines:
        if is_table_row(line) and units and is_table_row(units[-1][-1]):
            units[-1].append(line)
        else:
            units.append([line])
    return units


def split_unit(unit: list[str], room: int) -> list[list[str]]:
    # Split a unit that is too big on its own at its natural seams.
    if is_table_row(unit[0]):                     # a table: between rows, header repeated in every piece
        header, pieces = unit[0], [[unit[0]]]
        for row in unit[1:]:
            if len(pieces[-1]) > 1 and count_words("\n".join(pieces[-1] + [row])) > room:
                pieces.append([header])
            pieces[-1].append(row)
        return pieces
    sentences = [[]]                              # a paragraph: between sentences
    for sentence in SENTENCE_END.split(unit[0]):
        if sentences[-1] and count_words(" ".join(sentences[-1] + [sentence])) > room:
            sentences.append([])
        sentences[-1].append(sentence)
    return [[" ".join(s)] for s in sentences]     # a single sentence longer than `room` stays whole


def pack(units: list[list[str]], room: int) -> list[list[str]]:
    # Fill pieces with whole units, up to `room` words each.
    pieces, current = [], []
    for unit in units:
        if current and count_words("\n".join(current + unit)) > room:
            pieces.append(current)
            current = []
        if count_words("\n".join(unit)) <= room:
            current += unit
        else:
            *full, current = split_unit(unit, room)
            pieces += full
    if current:
        pieces.append(current)
    return pieces


def section_chunks(doc, max_words: int | None = None, context_header: bool = True) -> list[Chunk]:
    max_words = MAX_WORDS if max_words is None else max_words
    chunks = []
    for heading, lines in split_sections(doc.body):
        header = f"{doc.title} > {heading}" if context_header else ""
        pieces = pack(to_units(lines), room=max_words - count_words(header))
        for n, piece in enumerate(pieces, start=1):
            suffix = f"-part-{n}" if len(pieces) > 1 else ""
            text = "\n".join(([header] if header else []) + piece)
            chunks.append(make_chunk(doc, f"{doc.doc_id}#{slugify(heading)}{suffix}", heading, text))
    return chunks


structured = [c for d in documents for c in section_chunks(d)]
by_structured = {c.chunk_id: c for c in structured}
assert len(by_structured) == len(structured), "chunk IDs must be unique"

print(f"{len(documents)} documents -> {len(structured)} structure-aware chunks "
      f"(sections longer than {MAX_WORDS} words: {sum(c.chunk_id.endswith('-part-1') for c in structured)})\n")
show(by_structured["travel-policy#daily-limits-by-city"])

11 documents -> 44 structure-aware chunks (sections longer than 80 words: 0)

[travel-policy#daily-limits-by-city]  64 words
    Travel Policy > Daily limits by city
    Hotel and meal costs are reimbursed up to the daily limits below. Limits are per person and include taxes. Use the Other cities row for any city not listed.
    City | Hotel per night | Meals per day
    London | USD 250 | USD 75
    New York | USD 300 | USD 75
    Singapore | USD 220 | USD 60
    Other cities | USD 180 | USD 50



The whole table travels with its explanation, its column header and a first line that says which document and section it is from. With `MAX_WORDS = 80` no section here needed splitting.

### 4.1 When a section is too long

Real sections are often longer than any sensible chunk size. To see the splitting rules at work, cut the same section with a deliberately small limit of 30 words:

In [8]:
for chunk in section_chunks(travel, max_words=30):
    if chunk.section == "Daily limits by city":
        show(chunk)

[travel-policy#daily-limits-by-city-part-1]  25 words
    Travel Policy > Daily limits by city
    Hotel and meal costs are reimbursed up to the daily limits below. Limits are per person and include taxes.

[travel-policy#daily-limits-by-city-part-2]  16 words
    Travel Policy > Daily limits by city
    Use the Other cities row for any city not listed.

[travel-policy#daily-limits-by-city-part-3]  29 words
    Travel Policy > Daily limits by city
    City | Hotel per night | Meals per day
    London | USD 250 | USD 75
    New York | USD 300 | USD 75
    Singapore | USD 220 | USD 60

[travel-policy#daily-limits-by-city-part-4]  19 words
    Travel Policy > Daily limits by city
    City | Hotel per night | Meals per day
    Other cities | USD 180 | USD 50



Four pieces. The paragraph was split between sentences, the table between rows, and the last piece (`Other cities`) repeats `City | Hotel per night | Meals per day`, so its numbers keep their meaning. Each piece also repeats the context header: a few extra words per chunk, the price of self-contained chunks.

Piece 2 shows the limit of any splitter: *"Use the Other cities row for any city not listed"* refers to a table that is now in other pieces. Smaller chunks are not automatically more self-contained, which is one reason to keep the size limit generous.

## 5. Side by side — what each method does to the documents

Two structural checks, run on every document:

- **Lines cut in two**: sentences, list items or table rows that no single chunk contains in full.
- **Table rows cut off from their header**: data rows that share no chunk with their table's header row.

In [9]:
def tables(doc) -> list[list[str]]:
    return [unit for _, lines in split_sections(doc.body) for unit in to_units(lines) if is_table_row(unit[0])]


def structure_report(chunks: list[Chunk]) -> dict:
    by_doc_chunks = {}
    for c in chunks:
        by_doc_chunks.setdefault(c.doc_id, []).append(c.text)
    lines_cut = rows_orphaned = 0
    for d in documents:
        texts = by_doc_chunks[d.doc_id]
        for _, lines in split_sections(d.body):
            lines_cut += sum(not any(line in t for t in texts) for line in lines)
        for table in tables(d):
            header = table[0]
            rows_orphaned += sum(not any(row in t and header in t for t in texts) for row in table[1:])
    sizes = [count_words(c.text) for c in chunks]
    return {"chunks": len(chunks), "min": min(sizes), "median": statistics.median(sizes), "max": max(sizes),
            "indexed words": sum(sizes), "lines cut": lines_cut, "rows without header": rows_orphaned}


METHODS = {
    f"A. fixed {WINDOW_WORDS} words": fixed,
    "B. structure-aware": structured,
}
reports = {name: structure_report(chunks) for name, chunks in METHODS.items()}
table_rows = sum(len(t) - 1 for d in documents for t in tables(d))

print(f"{'':<20}{'chunks':>7}{'words per chunk':>18}{'indexed':>9}{'lines cut':>11}{'table rows cut off':>20}")
print(f"{'':<20}{'':>7}{'min  median  max':>18}{'words':>9}{'in two':>11}{'from their header':>20}")
for name, r in reports.items():
    print(f"{name:<20}{r['chunks']:>7}{r['min']:>6}{r['median']:>7g}{r['max']:>5}{r['indexed words']:>9}"
          f"{r['lines cut']:>11}{r['rows without header']:>13} of {table_rows}")

                     chunks   words per chunk  indexed  lines cut  table rows cut off
                             min  median  max    words     in two   from their header
A. fixed 40 words        42     3     40   40     1418         29            7 of 8
B. structure-aware       44    18     34   74     1557          0            0 of 8


About the same number of chunks and a similar typical size, so the comparison is fair on size. The difference is *where* the cuts fall:

- The 40-word windows cut 29 lines in two, and 7 of the 8 table rows end up in a chunk without their column header. In the software table the boundary falls right after the header row, so all four rows lose their column names.
- They also leave tiny leftovers: the last window of a document can be 3 words long.
- The structure-aware chunks cut nothing. They index more words in total because each one repeats its short context header.

## 6. Does it change what retrieval returns? 13 questions

Structure is only worth keeping if it changes results. Each question below comes with the **evidence**: the exact strings a chunk must contain to answer it *on its own*. For the Singapore question that is the row **and** the column header; a chunk with `USD 220` but no `Hotel per night` doesn't count.

Each question was chosen to probe one way chunking can fail, so the totals show *which failure modes* each method has. They are not an accuracy estimate for other documents or questions; Day 07 builds a proper labelled evaluation set, starting from these.

Both methods use exactly the same retriever (Day 01's BM25, top 3). Only the chunks differ.

In [10]:
@dataclass(frozen=True)
class Probe:
    question: str
    evidence: tuple[str, ...]    # strings that must ALL appear in one chunk for it to answer alone
    probes: str                  # what this question tests


QUESTIONS = [
    Probe("What is the hotel limit per night in Singapore?", ("Singapore | USD 220", "Hotel per night"),
          "PDF table: row + column header"),
    Probe("What is the daily meal limit in New York?", ("New York | USD 300 | USD 75", "Meals per day"),
          "PDF table: row + column header"),
    Probe("Who approves admin rights?", ("Admin rights | IT Security and your manager", "Who approves"),
          "HTML table: row + column header"),
    Probe("How long is parental leave?", ("16 weeks",), "Day 01: answer was at rank 3"),
    Probe("How many failed sign-in attempts lock my account?", ("five failed sign-in attempts",), "one sentence"),
    Probe("How long are application logs kept?", ("Application logs are kept for 13 months",), "one sentence"),
    Probe("How much is the home office allowance?", ("home office allowance of USD 500",), "one sentence"),
    Probe("How long do I have to claim the home office allowance?", ("within 60 days of approval",),
          "second sentence of a section"),
    Probe("Can I use my personal computer for work?", ("personal computers must not be used for company work",),
          "last sentence of a section"),
    Probe("Can I install software from the internet myself?", ("Do not install software from the internet yourself",),
          "text before the first heading"),
    Probe("What training must new starters complete?", ("security awareness and data protection courses",),
          "topic word only in the heading"),
    Probe("What happens after I report a security incident?", ("confirm receipt",),
          "topic words only in the heading"),
    Probe("How soon must I report a lost laptop?", ("within one hour",),
          "evidence spread over two sections"),
]


def evaluate(chunks: list[Chunk], k: int | None = None) -> list[dict]:
    # For each question: rank of the first retrieved chunk that holds ALL the evidence (None if none
    # does), whether the top k together hold it, and how many words a generator would have to read.
    k = TOP_K if k is None else k
    index = BM25Index({c.chunk_id: c.text for c in chunks})
    by_id = {c.chunk_id: c for c in chunks}
    results = []
    for probe in QUESTIONS:
        hits = index.search(probe.question, k=k)
        texts = [by_id[h.id].text for h in hits]
        results.append({
            "rank": next((h.rank for h, t in zip(hits, texts) if all(e in t for e in probe.evidence)), None),
            "across top k": all(any(e in t for t in texts) for e in probe.evidence),
            "context words": sum(count_words(t) for t in texts),
            "hits": [h.id for h in hits],
        })
    return results


def summary(results: list[dict]) -> str:
    one = sum(r["rank"] is not None for r in results)
    across = sum(r["across top k"] for r in results)
    words = statistics.mean(r["context words"] for r in results)
    return f"{one:>2}/{len(results)} in one chunk | {across:>2}/{len(results)} across top {TOP_K} | {words:5.0f} words read per question"


runs = {name: evaluate(chunks) for name, chunks in METHODS.items()}

mark = lambda r: str(r["rank"]) if r["rank"] else "-"
print(f"Rank of the first chunk holding the complete evidence ('-' = no top-{TOP_K} chunk holds it)\n")
print(f"{'question':<54}{'A':>3}{'B':>4}   probes")
for i, probe in enumerate(QUESTIONS):
    a, b = (runs[name][i] for name in METHODS)
    print(f"{probe.question:<54}{mark(a):>3}{mark(b):>4}   {probe.probes}")
print()
for name, results in runs.items():
    print(f"{name:<22} {summary(results)}")

Rank of the first chunk holding the complete evidence ('-' = no top-3 chunk holds it)

question                                                A   B   probes
What is the hotel limit per night in Singapore?         -   1   PDF table: row + column header
What is the daily meal limit in New York?               -   1   PDF table: row + column header
Who approves admin rights?                              -   1   HTML table: row + column header
How long is parental leave?                             1   3   Day 01: answer was at rank 3
How many failed sign-in attempts lock my account?       1   1   one sentence
How long are application logs kept?                     1   1   one sentence
How much is the home office allowance?                  1   1   one sentence
How long do I have to claim the home office allowance?  2   1   second sentence of a section
Can I use my personal computer for work?                1   1   last sentence of a section
Can I install software from the internet myself?

Same retriever, same questions, about the same number of words handed to the next step: the structure-aware chunks hold the complete evidence for all 13 questions, the fixed windows for 9.

The four misses are not random. Three are the table questions, where the row and its header ended up in different chunks. The fourth needs two sections. And the fixed windows are not worse everywhere: for *"How long is parental leave?"* their first window happens to hold the eligibility rule and the entitlement together and ranks first, while the structure-aware answer is at rank 3 (section 7.3 explains why).

Here is the Singapore question in detail:

In [11]:
def show_hits(question: str, chunks: list[Chunk]) -> None:
    index, by_id = BM25Index({c.chunk_id: c.text for c in chunks}), {c.chunk_id: c for c in chunks}
    for hit in index.search(question, k=TOP_K):
        text = by_id[hit.id].text.replace("\n", " / ")
        print(f"  #{hit.rank} {hit.score:5.2f}  [{hit.id}]  {text[:88]}{'...' if len(text) > 88 else ''}")


QUESTION = QUESTIONS[0].question
for name, chunks in METHODS.items():
    print(f"{name}: {QUESTION}")
    show_hits(QUESTION, chunks)
    print()

A. fixed 40 words: What is the hotel limit per night in Singapore?
  #1  9.01  [travel-policy#w080]  are reimbursed up to the daily limits below. Limits are per person and include taxes. Us...
  #2  3.29  [expense-reimbursement-policy#w120]  travel are reimbursed up to USD 75 per day. Alcohol is not reimbursed.
  #3  2.91  [travel-policy#w120]  300 | USD 75 / Singapore | USD 220 | USD 60 / Other cities | USD 180 | USD 50 /  / ## Cl...

B. structure-aware: What is the hotel limit per night in Singapore?
  #1 11.45  [travel-policy#daily-limits-by-city]  Travel Policy > Daily limits by city / Hotel and meal costs are reimbursed up to the dai...
  #2  2.97  [expense-reimbursement-policy#meals]  Expense Reimbursement Policy > Meals / Meals during business travel are reimbursed up to...
  #3  2.23  [remote-work-policy#eligibility]  Remote Work Policy > Eligibility / Employees may work remotely up to three days per week...



Method A's top chunk has the column headers but stops in the middle of the New York row. The Singapore row is in the rank-3 chunk, which starts with the leftover `300 | USD 75`. A model would have to stitch two fragments together and trust that the columns still line up. Method B's top chunk is the whole table, with its header and its context.

**Where structure-aware chunking is weakest.** Look at the last question: *"How soon must I report a lost laptop?"*. The answer, "within one hour", is in `security-incident-reporting#how-to-report`, but the words *lost laptop* appear in other sections: the definition of an incident and the travel policy's safety advice. So the section with the answer comes third. (The 40-word windows don't return it at all.) Questions whose evidence spans sections need bigger units (merging small neighbouring sections, exercise 4), retrieving the surrounding section or document around a hit (Day 11), or a generator that reads several chunks together (Day 06).

In [12]:
show_hits("How soon must I report a lost laptop?", structured)

  #1  7.42  [travel-policy#safety-while-travelling]  Travel Policy > Safety while travelling / Register every international trip in the trave...
  #2  4.76  [security-incident-reporting#what-counts-as-an-incident]  Security Incident Reporting > What counts as an incident / A security incident is any ev...
  #3  4.11  [security-incident-reporting#how-to-report]  Security Incident Reporting > How to report / Report suspected incidents to security@acm...


## 7. The knobs: overlap, window size and context headers

Three common pieces of advice, each measured on the same 13 questions.

### 7.1 Overlap

Overlap repeats the last words of each window at the start of the next, so a fact cut at a boundary appears whole in at least one window *if it is shorter than the overlap*.

In [13]:
overlapping = [c for d in documents for c in fixed_size_chunks(d, overlap=OVERLAP_WORDS)]
overlap_run = evaluate(overlapping)
extra = sum(count_words(c.text) for c in overlapping) / sum(count_words(c.text) for c in fixed) - 1

print(f"A. fixed {WINDOW_WORDS} words         {summary(runs[f'A. fixed {WINDOW_WORDS} words'])}")
print(f"A+ with {OVERLAP_WORDS}-word overlap   {summary(overlap_run)}")
print(f"\nOverlap indexes {len(overlapping)} chunks instead of {len(fixed)}: {extra:.0%} more words to embed and store.")
for probe, before, after in zip(QUESTIONS, runs[f"A. fixed {WINDOW_WORDS} words"], overlap_run):
    if before["rank"] is None or after["rank"] is None:
        status = "still missing" if after["rank"] is None else "repaired"
        print(f"  {status:<14} {probe.question}")

A. fixed 40 words          9/13 in one chunk | 11/13 across top 3 |   110 words read per question
A+ with 10-word overlap   12/13 in one chunk | 12/13 across top 3 |   112 words read per question

Overlap indexes 49 chunks instead of 42: 27% more words to embed and store.
  repaired       What is the hotel limit per night in Singapore?
  repaired       What is the daily meal limit in New York?
  still missing  Who approves admin rights?
  repaired       How soon must I report a lost laptop?


Overlap repaired three of the four misses, at a cost: a quarter more text to embed and store, and neighbouring windows that repeat each other, so the top 3 can contain the same passage twice. The software table is still cut off from its header: from the header row to the admin-rights row is 37 words, and none of the shifted windows happens to start in the right place to hold both. Overlap only rescues a fact when the fact *and the context it needs* fit inside the shared words. It is a patch for blind cuts, not a substitute for cutting in the right place.

### 7.2 Window size

If small windows cut tables, why not use bigger ones? Same questions, window sizes from 20 to 120 words, with and without 25 % overlap:

In [14]:
print(f"{'window':>7}{'overlap':>8}{'chunks':>8}{'complete in one chunk':>23}{'words read':>12}   Singapore row with its header?")
sweep = {}
for size in (20, 30, 40, 50, 60, 80, 100, 120):
    for overlap in (0, size // 4):
        chunks = [c for d in documents for c in fixed_size_chunks(d, size, overlap)]
        results = evaluate(chunks)
        complete = sum(r["rank"] is not None for r in results)
        words = statistics.mean(r["context words"] for r in results)
        kept = any("Singapore | USD 220" in c.text and "Hotel per night" in c.text for c in chunks)
        sweep[size, overlap] = {"complete": complete, "words": words, "singapore kept": kept}
        print(f"{size:>7}{overlap:>8}{len(chunks):>8}{complete:>19}/{len(QUESTIONS)}{words:>12.0f}   {'yes' if kept else 'no'}")

b_complete = sum(r["rank"] is not None for r in runs["B. structure-aware"])
b_words = statistics.mean(r["context words"] for r in runs["B. structure-aware"])
print(f"\nB. structure-aware (up to {MAX_WORDS} words): {b_complete}/{len(QUESTIONS)}, {b_words:.0f} words read")

 window overlap  chunks  complete in one chunk  words read   Singapore row with its header?
     20       0      76                  6/13          59   no
     20       5      96                  8/13          59   no
     30       0      52                  8/13          87   no
     30       7      64                  9/13          88   no
     40       0      42                  9/13         110   no
     40      10      49                 12/13         112   yes
     50       0      34                 12/13         139   yes
     50      12      40                 12/13         136   yes
     60       0      30                  9/13         166   no
     60      15      34                 11/13         163   yes
     80       0      23                 12/13         194   yes
     80      20      24                 13/13         213   yes
    100       0      21                 12/13         242   yes


    100      25      22                 13/13         238   yes
    120       0      18                 11/13         284   no


    120      30      18                 13/13         291   yes

B. structure-aware (up to 80 words): 13/13, 113 words read


Two lessons:

- **Bigger is not steadily better.** Without overlap, 50 words keeps the Singapore row with its header, 60 cuts them apart again, 80 and 100 keep them, 120 cuts them again. The result depends on where the boundaries happen to land, so it can change with any edit to the document.
- **Completeness bought with size is paid for in reading.** Large overlapping windows also reach 13/13, but each question then hands the next step about twice as many words, most of them about something else. More words cost more tokens, take longer, and give a language model more irrelevant text to be distracted by.

### 7.3 Context headers

Is the `Title > Section` line worth its words? Remove it and rerun:

In [15]:
headerless = [c for d in documents for c in section_chunks(d, context_header=False)]
headerless_run = evaluate(headerless)

print(f"B without headers     {summary(headerless_run)}")
print(f"B with headers        {summary(runs['B. structure-aware'])}\n")
for probe, without, with_header in zip(QUESTIONS, headerless_run, runs["B. structure-aware"]):
    if without["rank"] != with_header["rank"]:
        print(f"  rank {mark(without)} -> {mark(with_header)}   {probe.question}  ({probe.probes})")

B without headers     12/13 in one chunk | 12/13 across top 3 |    95 words read per question
B with headers        13/13 in one chunk | 13/13 across top 3 |   113 words read per question

  rank 2 -> 3   How long is parental leave?  (Day 01: answer was at rank 3)
  rank 2 -> 1   Can I install software from the internet myself?  (text before the first heading)
  rank - -> 1   What happens after I report a security incident?  (topic words only in the heading)


Headers help when the question's key words appear **only in a heading or title**: the *After you report* section never uses the words "report" or "incident" (it says "reported" and "Incidents", and there is no stemming). They can tip a close ranking: the software page's opening paragraph gains one more "software" from its title and moves from rank 2 to 1. They can also hurt: every chunk of the parental leave policy now starts with "Parental Leave Policy", so BM25 finds them harder to tell apart and the answer slips from rank 2 to rank 3. (The *Training* question didn't need the header: its text shares the word "complete" with the question.)

We keep the headers. A chunk that says where it comes from is easier for a reader, a citation or a language model to trust and use, and two questions improved against one that got worse, for about 18 more words read per question. But the retrieval effect is measured, not assumed, and it is mixed. Embedding models (Day 04) react to headers differently from BM25, so we'll measure again there.

## 8. Chunks inherit metadata; stable IDs make updates cheap

Every chunk carries what it needs to be **filtered, cited, dated and updated** without looking up its document:

In [16]:
record = asdict(by_structured["travel-policy#daily-limits-by-city"])
text = record.pop("text")
print(json.dumps(record, indent=2))

{
  "chunk_id": "travel-policy#daily-limits-by-city",
  "doc_id": "travel-policy",
  "section": "Daily limits by city",
  "title": "Travel Policy",
  "allowed_groups": [
    "all-employees"
  ],
  "source_path": "data/sample/acme-handbook/travel-policy.pdf",
  "last_reviewed": "2026-04-15",
  "doc_hash": "bf4bdf9b29cecca37f3f45b565a210bcda1f096d2a6be1fe5ca44d2ae7a90484",
  "chunk_hash": "e13a0fb0b50f3a0e13d6b7d61e6813fd6159faf7dd67958b6fccee0450a9e086"
}


`allowed_groups` comes straight from Day 02's permission export. The restricted manager guide shows why it must travel with every chunk: retrieval returns chunks, so a permission filter (Day 05, enforced on Day 14) can only work if each chunk knows who may see it.

In [17]:
manager_chunks = [c for c in structured if c.doc_id == "manager-pay-review-guide"]
print(f"manager-pay-review-guide -> {len(manager_chunks)} chunk(s), allowed_groups {manager_chunks[0].allowed_groups}\n")
for groups in ({"all-employees"}, {"all-employees", "managers"}):
    visible = [c for c in structured if groups & set(c.allowed_groups)]
    print(f"{sorted(groups)} -> {len(visible)} of {len(structured)} chunks")

manager-pay-review-guide -> 1 chunk(s), allowed_groups ('managers', 'people-team')

['all-employees'] -> 43 of 44 chunks
['all-employees', 'managers'] -> 44 of 44 chunks


**IDs and updates.** On Day 02, a changed `content_hash` told us a document had to be re-chunked. Re-chunking is cheap; **re-embedding** every chunk (Day 04) is the expensive part. With stable IDs and a hash per chunk, only chunks whose text really changed need new embeddings.

The demo below adds one sentence to the first section of the travel policy (in memory; the files are untouched), re-chunks the document both ways, and compares chunk by chunk:

In [18]:
NEW_SENTENCE = "This policy was updated in October 2026."
edited = replace(travel, body=travel.body.replace("## Purpose\n", f"## Purpose\n{NEW_SENTENCE}\n"))


def changed_chunks(before: list[Chunk], after: list[Chunk]) -> tuple[int, list[str]]:
    old = {c.chunk_id: c.chunk_hash for c in before}
    return len(after), [c.chunk_id for c in after if old.get(c.chunk_id) != c.chunk_hash]


print(f"Added {count_words(NEW_SENTENCE)} words at the start of travel-policy's Purpose section.\n")
for name, chunker in [("A. fixed 40 words", fixed_size_chunks), ("B. structure-aware", section_chunks)]:
    total, changed = changed_chunks(chunker(travel), chunker(edited))
    print(f"{name:<20} {len(changed)} of {total} chunks changed: {', '.join(c.split('#')[1] for c in changed)}")

Added 7 words at the start of travel-policy's Purpose section.

A. fixed 40 words    6 of 6 chunks changed: w000, w040, w080, w120, w160, w200
B. structure-aware   1 of 5 chunks changed: purpose


One sentence added at the top, and its seven new words push every window boundary along: all six chunks change, so all six must be re-embedded, and a citation such as `[travel-policy#w080]` now points to different text. The structure-aware chunks change in one place only, and `travel-policy#daily-limits-by-city` still means the daily limits.

There is a bonus for continuity: Day 03's IDs for the eight Markdown policies are exactly Day 01's chunk IDs (`remote-work-policy#core-hours` and so on), so Day 01's citations still resolve. A sanity check below enforces that.

Finally, write the chunks to disk for the next days. `outputs/` is in `.gitignore`.

In [19]:
OUT_DIR = REPO_ROOT / "outputs" / "day-03"
OUT_DIR.mkdir(parents=True, exist_ok=True)
with (OUT_DIR / "chunks.jsonl").open("w", encoding="utf-8", newline="\n") as f:
    for c in structured:
        f.write(json.dumps(asdict(c), ensure_ascii=False) + "\n")
print(f"wrote {len(structured)} chunks to {(OUT_DIR / 'chunks.jsonl').relative_to(REPO_ROOT).as_posix()}")

wrote 44 chunks to outputs/day-03/chunks.jsonl


## 9. Sanity checks

These assertions turn the behaviour above into quick, deterministic tests. `tests/test_day_03_notebook.py` runs this notebook in a fresh kernel, so any failing check fails the test. If you change the settings or the sample data, expect some to fail: read the message and update deliberately.

In [20]:
# Input: Day 02's records
assert len(documents) == 11 and len(quarantined) == 2
assert len(section_sizes) == 44 and (min(section_sizes), max(section_sizes)) == (13, 68)

# Method A cuts the Singapore row away from its header; method B keeps them together
assert "Hotel per night" not in singapore_fixed.text
assert "Singapore | USD 220 | USD 60" in by_structured["travel-policy#daily-limits-by-city"].text
assert reports["B. structure-aware"]["lines cut"] == 0 and reports["B. structure-aware"]["rows without header"] == 0
assert reports[f"A. fixed {WINDOW_WORDS} words"]["rows without header"] > 0

# Splitting a long section repeats the table header and keeps rows whole
small = [c for c in section_chunks(travel, max_words=30) if c.section == "Daily limits by city"]
assert len(small) == 4 and all(c.text.startswith("Travel Policy > Daily limits by city") for c in small)
assert all("City | Hotel per night | Meals per day" in c.text for c in small if " | " in c.text)

# Retrieval comparison on the 13 questions
complete = {name: sum(r["rank"] is not None for r in results) for name, results in runs.items()}
assert complete == {f"A. fixed {WINDOW_WORDS} words": 9, "B. structure-aware": 13}, complete
assert runs["B. structure-aware"][0]["rank"] == 1 and runs[f"A. fixed {WINDOW_WORDS} words"][0]["rank"] is None
assert sum(r["rank"] is not None for r in overlap_run) == 12
assert sum(r["rank"] is not None for r in headerless_run) == 12
kept = {size: sweep[size, 0]["singapore kept"] for size in (40, 50, 60, 80, 100, 120)}
assert kept == {40: False, 50: True, 60: False, 80: True, 100: True, 120: False}, kept   # section 7.2's text
assert sweep[80, 20]["complete"] == 13 and sweep[80, 20]["words"] > 1.8 * b_words
words_read = {name: round(statistics.mean(r["context words"] for r in results)) for name, results in
              [*runs.items(), ("B without headers", headerless_run)]}
assert words_read == {f"A. fixed {WINDOW_WORDS} words": 110, "B. structure-aware": 113,
                      "B without headers": 95}, words_read

# Metadata, IDs and updates
assert all(c.allowed_groups == by_doc[c.doc_id].allowed_groups for c in structured + fixed)
assert changed_chunks(fixed_size_chunks(travel), fixed_size_chunks(edited))[1].__len__() == 6
assert changed_chunks(section_chunks(travel), section_chunks(edited))[1] == ["travel-policy#purpose"]
day01_ids = {c.chunk_id for c in structured if by_doc[c.doc_id].source_format == "md"}
assert len(day01_ids) == 34 and "remote-work-policy#core-hours" in day01_ids   # Day 01's 34 chunk IDs

# Determinism
assert [asdict(c) for c in section_chunks(travel)] == [asdict(c) for c in section_chunks(travel)]

print("All Day 03 sanity checks passed.")

All Day 03 sanity checks passed.


## 10. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing settings.

1. **The boundary lottery.** Set `WINDOW_WORDS = 50`, rerun, and look at `boundary_map(travel, fixed_size_chunks(travel))`. Why does 50 keep the table together while 40 and 60 don't? Now simulate a policy update: add the 28-word note `"This version replaces the 2025 policy. Ask the Finance team if you are unsure which rules apply to a trip that was booked before this version was published."` to the *Purpose* section, the way section 8 adds its sentence. Does 50 still work? Do 40 and 60?
2. **Break the table rule.** In `to_units`, make every table row its own unit (delete the `and units and ...` condition). Run section 4.1 again. What does the `Other cities` piece lose?
3. **Your own heading-only question.** Find a section whose heading words don't appear in its text (hint: *Legal holds* vs "legal hold"; there is no stemming). Write a question, add it as a `Probe`, and compare B with and without headers.
4. **Merge small sections.** Write `merge_small_sections(chunks, max_words)` that joins neighbouring sections of the same document while they fit. Does it fix the lost-laptop question? What does it do to the parental leave question and to words read per question?
5. **Count tokens, not words.** If you have a tokenizer installed (for example the one for your embedding model), count tokens per chunk. How many tokens is `MAX_WORDS = 80` for these chunks? Which limit does your model actually enforce?

## Summary

- **Cut along the structure.** Split at headings, never inside a sentence or a table row, and repeat a table's header row when a table has to be split. On our 13 questions this kept the complete evidence in one chunk every time, at about the same size as 40-word windows, which managed 9.
- **Overlap and size are patches for blind cuts.** They fix some boundaries, miss or break others, and cost storage and prompt space. Measure them; don't assume.
- **A chunk is the unit of retrieval, citation, permission and update.** Give it a context header, its document's metadata and an ID that survives edits elsewhere in the document.

**Next — Day 04: embeddings, lexical matching and similarity.** These chunks become the input for dense retrieval, compared with today's BM25 on the same questions, including Day 01's *"time off for new dads"*.